# Filter with conditions · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 06. Filter with conditions

Build Boolean masks and combine conditions to answer precise questions.

Before you start: Know column selection and the basic comparison operators.

Official reference: https://pandas.pydata.org/docs/getting_started/intro_tutorials/03_subset_data.html

### M06C01. See the True and False values first

Comparing a Series with a value produces a Boolean Series, often called a mask. Each True marks a row that meets the condition. df.loc[mask, columns] keeps those rows and selects the requested columns. Keeping the mask in its own variable makes the logic easier to inspect.

**Read:** What is the type of value at each position in mask?

Answer: A Boolean value, True or False, for this clean non-missing Score column.

**Predict:** How many rows pass Score > 80?

Answer: Six: S001, S003, S006, S007, S010, and S012.

**Explain:** Why is df["Score"] > 80 not already the filtered table?

Answer: It only contains the condition results. A second selection uses that mask to retrieve the matching records.



In [ ]:
# Predict before running: How many rows pass Score > 80?
df = students()
mask = df["Score"] > 80
print(mask)
result = df.loc[mask, ["Student_ID", "Name", "Score"]]
print(result)

**Your turn:** Change the condition to Score < 70. Select Student_ID, Name, and Score into result. Print the number of matches.

For the check, keep these result names: `mask`, `result`.

**Hint:** Change > to < and use 70 as the threshold.

**Watch for this:** A mask should correspond to the table being filtered; a mask from another table can misalign.

In [ ]:
# Change this code to complete the task.
# Question: Change the condition to Score < 70. Select Student_ID, Name, and Score into result. Print the number of matches.
df = students()
mask = df["Score"] < 70
result = df.loc[mask, ["Student_ID", "Name", "Score"]]
print(result)
print("Matches:", len(result))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Student_ID'].tolist() == ['S004', 'S008', 'S011'] and mask.sum() == 3
print("Code checks passed. Review the written explanation as well.")

Dana, Hassan, and Khalid match the new rule, giving three rows. The question changed from high scores to scores needing attention.

**Quick check:** What does a True in a filter mask mean?

1. Keep that row.
2. Delete the column.
3. Replace the value with 1.

Answer: 1. A True marks a row to keep when the mask is used for selection.

**Explain what you learned:** I can separate the condition from the table selection it controls.

### M06C02. Combine conditions with AND and OR

Use & for element-by-element AND and | for element-by-element OR. Put parentheses around each comparison before combining them. AND requires both conditions; OR requires at least one. Python's and and or do not combine pandas Series masks this way.

**Read:** What two conditions must a row satisfy?

Answer: Its Campus must equal RAK and its Score must be at least 80.

**Predict:** Which names remain?

Answer: Amira, Chloe, and Grace.

**Explain:** What changes if & is replaced by |?

Answer: A student can pass either condition: be at RAK, have Score at least 80, or satisfy both. More rows can qualify.



In [ ]:
# Predict before running: Which names remain?
df = students()
mask = (df["Campus"] == "RAK") & (df["Score"] >= 80)
result = df.loc[mask, ["Student_ID", "Name", "Campus", "Score"]]
print(result)

**Your turn:** Select students who are at RAK OR have Score at least 90. Keep the same four output columns.

For the check, keep these result names: `result`.

**Hint:** Use | between two parenthesized comparisons.

**Watch for this:** Do not omit the parentheses around comparisons; operator precedence can change the expression.

In [ ]:
# Change this code to complete the task.
# Question: Select students who are at RAK OR have Score at least 90. Keep the same four output columns.
df = students()
mask = (df["Campus"] == "RAK") | (df["Score"] >= 90)
result = df.loc[mask, ["Student_ID", "Name", "Campus", "Score"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Student_ID'].tolist() == ['S001', 'S003', 'S005', 'S006', 'S007', 'S009', 'S011']
print("Code checks passed. Review the written explanation as well.")

All six RAK students qualify, and Fatima qualifies through her score of 95. Chloe meets both conditions but appears only once, so there are seven rows.

**Quick check:** Which correctly combines two pandas conditions with AND?

1. condition_a and condition_b
2. (condition_a) & (condition_b)
3. condition_a + "and" + condition_b

Answer: 2. Use & for element-wise AND; parenthesize each comparison when constructing the masks.

**Explain what you learned:** I can translate both, either, and at least into correct filter logic.

### M06C03. Membership, ranges, and NOT

isin() tests whether a value belongs to an allowed list. between() checks a range; inclusive="both" includes its two endpoints. The ~ operator reverses a Boolean mask. Naming masks can make a longer rule easier to explain.

**Read:** What does inclusive="both" tell between() to do?

Answer: Include values exactly equal to 70 or 90, as well as values between them.

**Predict:** Which AI students have scores between 70 and 90 here?

Answer: Amira, Grace, and Jamal.

**Explain:** Why does isin(["AI"]) use a list even for one allowed value?

Answer: isin expects a list-like collection of allowed values. The same expression can later include several categories.



In [ ]:
# Predict before running: Which AI students have scores between 70 and 90 here?
df = students()
in_program = df["Program"].isin(["AI"])
in_range = df["Score"].between(70, 90, inclusive="both")
result = df.loc[in_program & in_range, ["Name", "Program", "Score"]]
print(result)

**Your turn:** Select students whose Program is NOT AI and whose Score is between 70 and 85 inclusive. Print their names, programs, and scores.

For the check, keep these result names: `result`.

**Hint:** Reverse the membership mask with ~ and combine it with the new range mask.

**Watch for this:** Do not use Program == ["AI", "IT"] to test membership; use .isin(["AI", "IT"]).

In [ ]:
# Change this code to complete the task.
# Question: Select students whose Program is NOT AI and whose Score is between 70 and 85 inclusive. Print their names, programs, and scores.
df = students()
not_ai = ~df["Program"].isin(["AI"])
in_range = df["Score"].between(70, 85, inclusive="both")
result = df.loc[not_ai & in_range, ["Name", "Program", "Score"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Name'].tolist() == ['Bilal', 'Ethan', 'Ivy', 'Lina']
print("Code checks passed. Review the written explanation as well.")

Bilal, Ethan, Ivy, and Lina qualify. ~ reverses the AI membership test before the score range is applied.

**Quick check:** Which operator reverses a Boolean Series?

1. !
2. not
3. ~

Answer: 3. Use ~ for element-wise Boolean reversal of a pandas mask.

**Explain what you learned:** I can express allowed categories, numerical ranges, and excluded categories.

### M06C04. Read a filter written with query

query() is another way to express a row filter, using a short expression in a string. Column names appear inside that expression. @limit refers to a Python variable outside it. Learn the normal Boolean-mask form first so you can check what the query means.

**Read:** Why is there an @ before limit?

Answer: It tells query to use the Python variable limit instead of looking for a column named limit.

**Predict:** Which Dubai students meet the threshold?

Answer: Fatima and Jamal.

**Explain:** Why is and accepted here even though a previous cell used &?

Answer: query parses its expression string. Outside query, combining pandas Series directly requires the element-wise operators such as &.



In [ ]:
# Predict before running: Which Dubai students meet the threshold?
df = students()
limit = 85
result = df.query("Score >= @limit and Campus == 'Dubai'")[["Name", "Campus", "Score"]]
print(result)

**Your turn:** Use a normal Boolean mask instead of query to find Dubai students with Score at least 80. Save the same three columns in result.

For the check, keep these result names: `result`.

**Hint:** Combine (df["Score"] >= limit) and (df["Campus"] == "Dubai") with &.

**Watch for this:** Use fixed, trusted query expressions for these lessons; do not construct executable expressions from arbitrary user text.

In [ ]:
# Change this code to complete the task.
# Question: Use a normal Boolean mask instead of query to find Dubai students with Score at least 80. Save the same three columns in result.
df = students()
limit = 80
mask = (df["Score"] >= limit) & (df["Campus"] == "Dubai")
result = df.loc[mask, ["Name", "Campus", "Score"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Name'].tolist() == ['Fatima', 'Jamal', 'Lina']
print("Code checks passed. Review the written explanation as well.")

Fatima, Jamal, and Lina meet the threshold of 80. A mask and a query can express the same selection rule.

**Quick check:** Inside query, what does @limit refer to?

1. A Python variable named limit.
2. Always a column called limit.
3. The last row.

Answer: 1. @ accesses a variable from the surrounding Python scope within the query expression.

**Explain what you learned:** I can translate a simple query into a Boolean mask and explain both.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates